# Peru Black Marble example with `panelplots`

This notebook template loads the Peru daily Black Marble panel from a Hugging Face dataset repository and produces two standardized figures:

1. A spaghetti plot.
2. A descriptive decile fan chart.

The example assumes a private repository and a token stored as a Colab secret named `HF_Writing`, but the plotting code is independent of Hugging Face once `df_daily` exists.

In [ ]:
# Install the package from GitHub after publishing the repository.

# !pip install git+https://github.com/Favioleiva/panelplots.git

# For local development inside the cloned repository, use instead:
# !pip install -e .

In [ ]:
# Core imports

import os
import re
from pathlib import Path

import pandas as pd

from huggingface_hub import HfApi, hf_hub_download, login, whoami
from panelplots import plot_fanchart, plot_spaghetti

print("Imports ready.")

## 1. Hugging Face configuration

Edit these values if the repository or folder structure changes.

In [ ]:
HF_REPO_ID = "faviolc/BlackMarble"
HF_REPO_TYPE = "dataset"

COUNTRY_SLUG = "peru"
PRODUCT = "vnp46a2"
COLLECTION = "5200"
LABEL = "main_gapfilled"

HF_BASE_PATH = f"{COUNTRY_SLUG}/daily_raw/{PRODUCT}_c{COLLECTION}/{LABEL}"
HF_MONTHLY_PARQUET_DIR = f"{HF_BASE_PATH}/monthly_parquet"

LOCAL_ROOT = Path("/content/blackmarble_peru_from_hf")
LOCAL_MONTHLY_DIR = LOCAL_ROOT / "monthly_parquet"
LOCAL_OUTPUT_DIR = LOCAL_ROOT / "figures"

LOCAL_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Monthly Parquet folder:", HF_MONTHLY_PARQUET_DIR)
print("Local output folder:", LOCAL_OUTPUT_DIR)

## 2. Authenticate

The notebook first tries to read a Colab secret named `HF_Writing`. If that does not exist, it tries the environment variable `HF_Writing`.

In [ ]:
HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_Writing")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_Writing")

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Paste a Hugging Face token with read access: ")

login(token=HF_TOKEN, add_to_git_credential=True)
api = HfApi(token=HF_TOKEN)

print("Logged in as:", whoami(token=HF_TOKEN).get("name"))

## 3. List and download monthly Parquets

In [ ]:
all_repo_files = api.list_repo_files(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    token=HF_TOKEN,
)

monthly_pattern = re.compile(
    r"BlackMarble_VNP46A2_Daily_Peru_main_gapfilled_\d{4}_\d{2}\.parquet$"
)

monthly_parquet_files = sorted(
    path for path in all_repo_files
    if path.startswith(HF_MONTHLY_PARQUET_DIR + "/")
    and path.endswith(".parquet")
    and monthly_pattern.search(Path(path).name)
)

if not monthly_parquet_files:
    raise FileNotFoundError(f"No monthly Parquet checkpoints found under {HF_MONTHLY_PARQUET_DIR}")

print("Monthly files found:", len(monthly_parquet_files))
print("First file:", monthly_parquet_files[0])
print("Last file:", monthly_parquet_files[-1])

In [ ]:
local_files = []

for remote_path in monthly_parquet_files:
    local_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        filename=remote_path,
        token=HF_TOKEN,
        local_dir=str(LOCAL_MONTHLY_DIR),
        local_dir_use_symlinks=False,
    )
    local_files.append(local_path)

print("Downloaded or cached files:", len(local_files))

## 4. Combine the panel

In [ ]:
monthly_frames = []

for local_path in local_files:
    df_month = pd.read_parquet(local_path)
    df_month["source_file"] = Path(local_path).name
    monthly_frames.append(df_month)

df_daily = pd.concat(monthly_frames, ignore_index=True)

df_daily["date"] = pd.to_datetime(df_daily["date"]).dt.normalize()
df_daily["ID_ADMIN"] = df_daily["ID_ADMIN"].astype(str)
df_daily = df_daily.sort_values(["ID_ADMIN", "date"]).reset_index(drop=True)

print("Rows:", f"{len(df_daily):,}")
print("Columns:", f"{df_daily.shape[1]:,}")
print("Districts:", df_daily["ID_ADMIN"].nunique())
print("Date range:", df_daily["date"].min(), "to", df_daily["date"].max())

## 5. Spaghetti plot

This reproduces the Peru version using `log(1 + 1000 × mean daily radiance)` and a fixed y-axis from 0 to 16.

In [ ]:
fig, ax = plot_spaghetti(
    df_daily,
    y="mean",
    time="date",
    unit="ID_ADMIN",
    transform="blackmarble_log_mean",
    transform_scale=1000,
    expected_units=1793,
    y_label="Log
NTL",
    x_label="Year",
    y_limits=(0, 16),
    y_tick_step=2,
    mean_label="Mean across 1,793 districts",
    output_path=LOCAL_OUTPUT_DIR / "peru_daily_spaghetti_log_mean_x1000_plus1_y0_16.png",
    output_formats=["png", "pdf", "svg"],
    return_data=False,
)

## 6. Descriptive decile fan chart

In [ ]:
fig, ax, fan_stats, saved_paths = plot_fanchart(
    df_daily,
    y="mean",
    time="date",
    unit="ID_ADMIN",
    transform="blackmarble_log_mean",
    transform_scale=1000,
    expected_units=1793,
    y_label="Log
NTL",
    x_label="Year",
    y_limits=(0, 16),
    y_tick_step=2,
    output_path=LOCAL_OUTPUT_DIR / "peru_daily_decile_fanchart_log_mean_x1000_plus1_y0_16.png",
    output_formats=["png", "pdf", "svg"],
    stats_output_path=LOCAL_OUTPUT_DIR / "peru_daily_decile_fanchart_quantiles_log_mean_x1000_plus1.csv",
    return_data=True,
)

fan_stats.head()

## 7. Footnote text

Use or adapt this text in slides, papers, or thesis figures.

In [ ]:
spaghetti_note = (
    "Note: Thin colored lines show daily trajectories for individual Peruvian districts. "
    "Colors follow long-run district brightness using the plasma palette. The thick line "
    "shows the simple daily average across districts. Transformation: log(1 + 1000 × mean "
    "daily radiance). The y-axis is fixed from 0 to 16 to make figures directly comparable "
    "across countries. Blank vertical gaps correspond to days with no valid district-level "
    "observations after Black Marble quality filtering and zonal statistics."
)

fanchart_note = (
    "Note: The decile fan chart summarizes the daily cross-sectional distribution of "
    "Peruvian districts. Bands show adjacent decile intervals of the transformed outcome, "
    "from the lower tail in blue tones to the upper tail in warm orange-red tones; middle "
    "deciles are shown in yellow. Transformation: log(1 + 1000 × mean daily radiance). "
    "The y-axis is fixed from 0 to 16 to make the Peru, Indonesia, and Bolivia figures "
    "directly comparable. Blank vertical gaps correspond to days with no valid district-level "
    "observations after Black Marble quality filtering and zonal statistics."
)

print(spaghetti_note)
print()
print(fanchart_note)